[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/solutions/exercises/30_sequence_models/90_exercises.ipynb)

# Bài tập — Chương 30: Sequence Models (L0 → L7)

> 📗 **BẢN LỜI GIẢI** — chỉ mở sau khi đã tự làm bài. Bản bài tập (đã xoá lời giải) nằm trong thư mục `notebooks/` của chương.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/solutions/exercises/30_sequence_models

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

from ml_course.checks import check_close, check_equal, check_greater, check_less, check_true
from ml_course.utils import repo_path

try:
    import torch
    from torch import nn
    from torch.nn.utils.rnn import pack_padded_sequence
except ImportError as e:
    raise ImportError("Bài tập này cần PyTorch: pip install -e \".[dl]\"") from e

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.deterministic = True
rng = np.random.default_rng(0)

## L0 — Nhớ khái niệm

**Câu 1.** (a) Công thức cập nhật trạng thái ẩn của RNN; vì sao số tham số không phụ thuộc độ dài chuỗi? (b) Vì sao RNN khó học phụ thuộc xa? (c) Các phương trình LSTM; vai trò cổng quên. (d) Teacher forcing là gì? (e) Công thức dot-product attention và lợi ích so với seq2seq một vector ngữ cảnh.

✍️ Trả lời:

**Đáp án:**
(a) $\mathbf h_t = \tanh(W_{xh}\mathbf x_t + W_{hh}\mathbf h_{t-1} + \mathbf b)$; cùng bộ trọng số dùng ở mọi bước thời gian.
(b) Gradient qua thời gian là tích các Jacobian $\text{diag}(1 - \mathbf h_t^2)W_{hh}$ → tắt dần (hoặc bùng nổ) theo hàm mũ với khoảng cách.
(c) $\mathbf f, \mathbf i, \mathbf o = \sigma(\cdot)$, $\mathbf g = \tanh(\cdot)$, $\mathbf c_t = \mathbf f\odot\mathbf c_{t-1} + \mathbf i\odot\mathbf g$, $\mathbf h_t = \mathbf o\odot\tanh\mathbf c_t$. Cổng quên quyết định giữ bao nhiêu ký ức; $\mathbf f \approx 1$ cho gradient đi qua gần nguyên vẹn.
(d) Khi train, decoder nhận token **đúng** của bước trước làm đầu vào (thay vì dự đoán của chính nó) → train song song, ổn định; đổi lại có exposure bias khi suy luận.
(e) $\alpha_{tj} = \text{softmax}_j(\mathbf s_t^\top\mathbf h_j)$, $\mathbf c_t = \sum_j\alpha_{tj}\mathbf h_j$; decoder truy cập mọi trạng thái encoder thay vì một vector nén → không còn nút cổ chai, tự học căn chỉnh.

## L1 — Giải thích bằng lời

**Câu 2.** Một nhóm dự báo giá cổ phiếu ngày mai bằng **LSTM hai chiều** đọc cửa sổ 60 ngày (gồm cả ngày cần dự báo, được che bằng 0), đạt sai số validation rất thấp khi chia train/validation **ngẫu nhiên** theo cửa sổ. Khi chạy thật thì kết quả kém hẳn. Chỉ ra các vấn đề.

✍️ Trả lời:

**Gợi ý:** (1) LSTM **hai chiều** đọc chuỗi theo cả chiều ngược — trạng thái tại một vị trí chứa thông tin từ các bước sau; nếu cửa sổ chứa dữ liệu sau thời điểm dự báo (hoặc đặc trưng tính từ tương lai) thì đó là rò rỉ. Dự báo/sinh tuần tự phải dùng mô hình **một chiều** (nhân quả). (2) Chia ngẫu nhiên các cửa sổ chồng lấn → cửa sổ validation gần như trùng cửa sổ train (trùng 59/60 ngày) → đánh giá quá lạc quan; phải chia theo thời gian / backtest (chương 25). (3) Giá cổ phiếu gần bước ngẫu nhiên: cần so với baseline naive (giá hôm nay); dự báo lợi suất thay vì mức giá; cẩn thận với chuẩn hoá dùng thống kê toàn chuỗi (rò rỉ).

## L2 — Tính tay

**Câu 3.** RNN một chiều (vô hướng): $h_t = \tanh(w\,h_{t-1} + u\,x_t)$, $w = 0.5$, $u = 1$, $h_0 = 0$, $\mathbf x = [1, 0, -1]$. Tính $h_1, h_2, h_3$ và $\partial h_3/\partial h_1 = \prod_{t=2}^{3}w(1 - h_t^2)$.

In [ ]:
### START CODE HERE ### (≈ 4 dòng)
h1 = math.tanh(1.0)
h2 = math.tanh(0.5 * h1)
h3 = math.tanh(0.5 * h2 - 1.0)
dh3_dh1 = 0.5 * (1 - h2**2) * 0.5 * (1 - h3**2)
### END CODE HERE ###
print(f"h = {h1:.4f}, {h2:.4f}, {h3:.4f} | ∂h3/∂h1 = {dh3_dh1:.4f}")

In [ ]:
th = torch.tensor(0.0, requires_grad=True)
hs = [th]
for x in [1.0, 0.0, -1.0]:
    hs.append(torch.tanh(0.5 * hs[-1] + x))
hs[1].retain_grad()
hs[3].backward()
check_close([h1, h2, h3], [hs[1].item(), hs[2].item(), hs[3].item()], tol=1e-6, message="trạng thái ẩn")
check_close(dh3_dh1, hs[1].grad.item(), tol=1e-6, message="∂h3/∂h1 khớp autograd")
check_less(dh3_dh1, 0.25, "mỗi bước nhân ≤ w = 0.5 → gradient co lại")

**Câu 4.** (a) Một bước LSTM vô hướng: $c_{t-1} = 1$, các cổng đã tính $f = 0.9$, $i = 0.2$, $g = 0.5$, $o = 0.8$. Tính $c_t$, $h_t$. (b) Số tham số của `nn.RNN`, `nn.GRU`, `nn.LSTM` với input 10, hidden 20 (1 lớp, có bias).

In [ ]:
### START CODE HERE ### (≈ 5 dòng)
c_t = 0.9 * 1 + 0.2 * 0.5
h_t = 0.8 * math.tanh(c_t)
n_rnn = 20 * 10 + 20 * 20 + 2 * 20
n_gru = 3 * n_rnn
n_lstm = 4 * n_rnn
### END CODE HERE ###
print(f"c_t = {c_t}, h_t = {h_t:.4f} | RNN {n_rnn}, GRU {n_gru}, LSTM {n_lstm}")

In [ ]:
check_close([c_t, h_t], [1.0, 0.8 * math.tanh(1.0)], message="bước LSTM")
for n, cls in [(n_rnn, nn.RNN), (n_gru, nn.GRU), (n_lstm, nn.LSTM)]:
    check_equal(n, sum(p.numel() for p in cls(10, 20).parameters()), f"số tham số {cls.__name__}")

## L3 — Code

**Bài 5.** Viết `lstm_forward(x_seq, h0, c0, W_ih, W_hh, b_ih, b_hh)` (NumPy, thứ tự cổng PyTorch `i, f, g, o`) trả về `(outputs, h_T, c_T)` với `outputs` shape (T, H). So khớp `nn.LSTM`.

In [ ]:
def lstm_forward(x_seq, h0, c0, W_ih, W_hh, b_ih, b_hh):
    sig = lambda z: 1 / (1 + np.exp(-z))
    h, c, outs = h0, c0, []
    ### START CODE HERE ### (≈ 8 dòng)
    # STARTER: # với mỗi x trong x_seq: gates = W_ih x + b_ih + W_hh h + b_hh; tách 4 phần (i, f, g, o)
    # STARTER: # c = f*c + i*g; h = o*tanh(c); lưu h
    for x in x_seq:
        gates = W_ih @ x + b_ih + W_hh @ h + b_hh
        i, f, g, o = np.split(gates, 4)
        i, f, g, o = sig(i), sig(f), np.tanh(g), sig(o)
        c = f * c + i * g
        h = o * np.tanh(c)
        outs.append(h)
    ### END CODE HERE ###
    return np.array(outs), h, c

In [ ]:
torch.manual_seed(0)
lstm5 = nn.LSTM(4, 6).double()
W5 = {k: v.detach().numpy() for k, v in lstm5.state_dict().items()}
x5 = torch.randn(9, 1, 4, dtype=torch.float64)
h0 = torch.randn(1, 1, 6, dtype=torch.float64)
c0 = torch.randn(1, 1, 6, dtype=torch.float64)
out_ref, (hT_ref, cT_ref) = lstm5(x5, (h0, c0))
outs, hT, cT = lstm_forward(x5[:, 0].numpy(), h0[0, 0].numpy(), c0[0, 0].numpy(),
                            W5["weight_ih_l0"], W5["weight_hh_l0"], W5["bias_ih_l0"], W5["bias_hh_l0"])
check_close(outs, out_ref[:, 0].detach().numpy(), tol=1e-10, message="đầu ra mọi bước")
check_close(cT, cT_ref[0, 0].detach().numpy(), tol=1e-10, message="trạng thái ô cuối")

**Bài 6.** Viết `attention(query, keys, values, mask=None, scale=True)`: `query` (N, Tq, d), `keys` (N, Tk, d), `values` (N, Tk, dv); điểm = $QK^\top$ (chia $\sqrt d$ nếu `scale`), vị trí `mask == False` bị gán $-\infty$ trước softmax; trả về `(output, weights)`. So với `torch.nn.functional.scaled_dot_product_attention`.

In [ ]:
def attention(query, keys, values, mask=None, scale=True):
    ### START CODE HERE ### (≈ 6 dòng)
    # STARTER: return None, None
    scores = query @ keys.transpose(-2, -1)
    if scale:
        scores = scores / math.sqrt(query.size(-1))
    if mask is not None:
        scores = scores.masked_fill(~mask, float("-inf"))
    weights = torch.softmax(scores, dim=-1)
    return weights @ values, weights
    ### END CODE HERE ###

In [ ]:
torch.manual_seed(1)
Q, K, Vv = torch.randn(2, 3, 8), torch.randn(2, 5, 8), torch.randn(2, 5, 4)
mask6 = torch.ones(2, 3, 5, dtype=torch.bool)
mask6[0, :, 3:] = False                                                  # chuỗi 0 chỉ có 3 vị trí thật
out6, w6 = attention(Q, K, Vv, mask6)
ref6 = torch.nn.functional.scaled_dot_product_attention(Q, K, Vv, attn_mask=mask6)
check_close(out6.numpy(), ref6.numpy(), tol=1e-6, message="khớp scaled_dot_product_attention")
check_close(w6.sum(-1).numpy(), np.ones((2, 3)), tol=1e-6, message="trọng số mỗi hàng tổng = 1")
check_close(w6[0, :, 3:].numpy(), np.zeros((3, 2)), message="không chú ý vào vị trí bị che")

## L4 — Debug

**Bài 7.** Phân loại chuỗi độ dài thay đổi (có đệm `PAD = 0` ở cuối). Hàm `last_hidden` có lỗi: lấy `out[:, -1]` — với chuỗi ngắn, đó là trạng thái **sau khi đọc phần đệm**, không phải tại token thật cuối cùng. Sửa `last_hidden(out, lens)` để lấy trạng thái tại vị trí `lens - 1` của mỗi chuỗi (dùng `gather` hoặc chỉ số nâng cao). Kiểm tra: kết quả phải khớp trạng thái cuối khi dùng `pack_padded_sequence`.

In [ ]:
def last_hidden(out, lens):
    ### START CODE HERE ### (≈ 2 dòng)
    # STARTER: return out[:, -1]                                                      # 🐛 trạng thái sau phần đệm
    idx = (lens - 1).to(out.device)
    return out[torch.arange(out.size(0), device=out.device), idx]
    ### END CODE HERE ###

In [ ]:
torch.manual_seed(2)
gru7 = nn.GRU(3, 8, batch_first=True)
lens7 = torch.tensor([5, 2, 7, 3])
x7 = torch.randn(4, 7, 3)
for i, l in enumerate(lens7.tolist()):
    x7[i, l:] = 0.0                                                     # phần đệm
out7, _ = gru7(x7)
_, h_packed = gru7(pack_padded_sequence(x7, lens7, batch_first=True, enforce_sorted=False))
check_close(last_hidden(out7, lens7).detach().numpy(), h_packed[0].detach().numpy(), tol=1e-6, message="khớp trạng thái cuối của chuỗi đã pack")
check_true(not torch.allclose(out7[1, -1], h_packed[0, 1]), "out[:, -1] của chuỗi ngắn KHÁC trạng thái đúng (đã đọc đệm)")

## L5 — Phân tích

**Bài 8.** **Độ dài phụ thuộc.** Adding problem (bài 03) với $T \in \{10, 30\}$: train `nn.RNN` và `nn.GRU` (hidden 64, bias cổng cập nhật GRU = 1), 2 000 bước, Adam lr 1e-3, clip 1.0, batch 64; lưu `mse8[(kind, T)]` = MSE trên 1 000 mẫu validation. Nhận xét.

In [ ]:
def adding_batch(n, T, g):
    values = torch.rand(n, T, generator=g)
    marks = torch.zeros(n, T)
    a = torch.randint(0, T // 2, (n,), generator=g)
    b = torch.randint(T // 2, T, (n,), generator=g)
    marks[torch.arange(n), a] = 1
    marks[torch.arange(n), b] = 1
    return torch.stack([values, marks], dim=-1), (values * marks).sum(1)


class SeqRegressor(nn.Module):
    def __init__(self, kind, hidden=64):
        super().__init__()
        self.rnn = {"RNN": nn.RNN, "GRU": nn.GRU}[kind](2, hidden, batch_first=True)
        self.head = nn.Linear(hidden, 1)
        if kind == "GRU":
            with torch.no_grad():
                self.rnn.bias_ih_l0[hidden:2 * hidden].fill_(1.0)

    def forward(self, x):
        out, _ = self.rnn(x)
        return self.head(out[:, -1]).squeeze(1)


mse8 = {}
### START CODE HERE ### (≈ 14 dòng)
# STARTER: # với T trong [10, 30], kind trong ["RNN", "GRU"]: torch.manual_seed(0); g = Generator(seed 0);
# STARTER: # validation = adding_batch(1000, T, Generator(seed 100)); 2 000 bước train như mô tả; mse8[(kind, T)] = MSE validation
for T in [10, 30]:
    for kind in ["RNN", "GRU"]:
        torch.manual_seed(0)
        g = torch.Generator().manual_seed(0)
        X_val, y_val = adding_batch(1000, T, torch.Generator().manual_seed(100))
        model = SeqRegressor(kind).to(device)
        opt = torch.optim.Adam(model.parameters(), lr=1e-3)
        for _ in range(2000):
            xb, yb = adding_batch(64, T, g)
            opt.zero_grad()
            nn.functional.mse_loss(model(xb.to(device)), yb.to(device)).backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        with torch.no_grad():
            mse8[(kind, T)] = nn.functional.mse_loss(model(X_val.to(device)), y_val.to(device)).item()
### END CODE HERE ###
print(pd.Series(mse8).unstack().round(4), "\n(dự đoán hằng số ≈ 0.167)")

In [ ]:
check_less(mse8[("RNN", 10)], 0.08, "RNN học được khi phụ thuộc ngắn (T = 10)")
check_greater(mse8[("RNN", 30)], 0.12, "RNN thất bại khi T = 30")
check_less(mse8[("GRU", 30)], 0.02, "GRU giải được T = 30")

**Nhận xét mẫu:** Với $T = 10$, khoảng cách tối đa giữa số được đánh dấu đầu tiên và cuối chuỗi ~10 bước — RNN vẫn truyền đủ gradient để học. Với $T = 30$, gradient tới nửa đầu chuỗi đã tắt (bài 02) → RNN mắc kẹt ở mức dự đoán hằng số. GRU, nhờ cổng cập nhật giữ trạng thái (bias = 1 → mặc định "nhớ"), giải được cả hai. Ranh giới "ngắn/dài" phụ thuộc kích thước ẩn, khởi tạo, số bước train — nhưng xu hướng thì luôn như vậy.

## L6 — Thực tế

**Bài 9.** **Phát hiện chuỗi sự kiện đáng ngờ.** Mỗi phiên là chuỗi 10–30 sự kiện (8 loại); mọi phiên đều có `đăng nhập thiết bị lạ` (mã 6) và `đổi mật khẩu` (mã 7). Phiên **đáng ngờ** (nhãn 1) nếu **đổi mật khẩu xảy ra sau** đăng nhập thiết bị lạ. (1) Baseline: `LogisticRegression` trên **số lần xuất hiện** mỗi loại sự kiện (bag-of-events) → `acc_bow`. (2) GRU + embedding + `pack_padded_sequence` → `acc_gru` ≥ 0.95. Giải thích chênh lệch.

In [ ]:
def make_sessions(n, seed):
    r = np.random.default_rng(seed)
    seqs, labels = [], []
    for _ in range(n):
        L = r.integers(10, 31)
        s = [int(v) for v in r.integers(0, 6, L)]                         # sự kiện thường 0–5
        p1, p2 = sorted(r.choice(L + 2, 2, replace=False))
        suspicious = r.random() < 0.5
        first, second = (6, 7) if suspicious else (7, 6)
        s.insert(p1, first)
        s.insert(p2, second)
        seqs.append(s)
        labels.append(int(suspicious))
    return seqs, np.array(labels)


train_seqs, y9_tr = make_sessions(3000, seed=0)
test_seqs, y9_te = make_sessions(1000, seed=1)
print("ví dụ:", train_seqs[0], "→ nhãn", y9_tr[0])
### START CODE HERE ### (≈ 30 dòng)
# STARTER: # (1) đếm: counts[i, k] = số lần sự kiện k trong phiên i → LogisticRegression → acc_bow
# STARTER: # (2) pad về cùng độ dài (mã đệm = 8, Embedding(9, 16, padding_idx=8)), lens; GRU(16, 32) + Linear(32, 2);
# STARTER: #     dùng h cuối từ pack_padded_sequence; Adam 3e-3, 10 epoch batch 64 → acc_gru
# STARTER: acc_bow, acc_gru = None, None
def counts(seqs):
    return np.array([np.bincount(s, minlength=8) for s in seqs])


acc_bow = LogisticRegression(max_iter=2000).fit(counts(train_seqs), y9_tr).score(counts(test_seqs), y9_te)


def pad(seqs):
    lens = torch.tensor([len(s) for s in seqs])
    X = torch.full((len(seqs), int(lens.max())), 8, dtype=torch.long)
    for i, s in enumerate(seqs):
        X[i, :len(s)] = torch.tensor(s)
    return X, lens


class SessionGRU(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(9, 16, padding_idx=8)
        self.gru = nn.GRU(16, 32, batch_first=True)
        self.out = nn.Linear(32, 2)

    def forward(self, x, lens):
        _, h = self.gru(pack_padded_sequence(self.emb(x), lens, batch_first=True, enforce_sorted=False))
        return self.out(h[-1])


Xtr9, Ltr9 = pad(train_seqs)
Xte9, Lte9 = pad(test_seqs)
ytr9, yte9 = torch.tensor(y9_tr), torch.tensor(y9_te)
torch.manual_seed(0)
model9 = SessionGRU().to(device)
opt9 = torch.optim.Adam(model9.parameters(), lr=3e-3)
g9 = torch.Generator().manual_seed(0)
for epoch in range(10):
    model9.train()
    perm = torch.randperm(len(Xtr9), generator=g9)
    for s in range(0, len(perm), 64):
        idx = perm[s:s + 64]
        opt9.zero_grad()
        nn.functional.cross_entropy(model9(Xtr9[idx].to(device), Ltr9[idx]), ytr9[idx].to(device)).backward()
        opt9.step()
model9.eval()
with torch.no_grad():
    acc_gru = (model9(Xte9.to(device), Lte9).argmax(1).cpu() == yte9).float().mean().item()
### END CODE HERE ###
print(f"bag-of-events + LogReg = {acc_bow:.3f} | GRU = {acc_gru:.3f}")

In [ ]:
check_less(acc_bow, 0.6, "bag-of-events gần như đoán mò")
check_greater(acc_gru, 0.95, "GRU nắm được thứ tự")

**Nhận xét:** Mọi phiên đều chứa đúng một lần mỗi sự kiện 6 và 7, phần còn lại ngẫu nhiên → vector đếm **giống hệt về phân phối** giữa hai lớp; thông tin phân biệt nằm hoàn toàn ở **thứ tự**. Bag-of-events (và mọi mô hình bỏ thứ tự) không thể làm tốt hơn đoán mò; GRU đọc tuần tự, "ghi nhớ" đã thấy sự kiện 6 hay chưa khi gặp sự kiện 7. Thực tế (an ninh, gian lận, hành trình khách hàng): khi thứ tự/khoảng cách thời gian giữa sự kiện mang tín hiệu, cần mô hình chuỗi — hoặc feature engineering mã hoá thứ tự (n-gram sự kiện, "A trước B", thời gian từ A đến B).

## L7 — Nâng cao

**Bài 10.** **Beam search** cho mô hình ngôn ngữ ký tự. Huấn luyện nhanh một GRU char-LM trên review (cho sẵn). Viết `beam_search(model, start, length, beam_width)` → `(chuỗi, tổng log-xác suất)`: ở mỗi bước mở rộng mọi chùm bằng mọi ký tự, giữ `beam_width` chuỗi có tổng log-prob cao nhất. So với greedy (= beam width 1).

In [ ]:
reviews = pd.read_csv(repo_path("data", "sample", "reviews_vi.csv"))["text"].tolist()
corpus = "\n".join(reviews) + "\n"
chars = sorted(set(corpus))
stoi = {c: i for i, c in enumerate(chars)}
data = torch.tensor([stoi[c] for c in corpus])


class CharGRU(nn.Module):
    def __init__(self, vocab, emb=32, hidden=128):
        super().__init__()
        self.emb = nn.Embedding(vocab, emb)
        self.gru = nn.GRU(emb, hidden, batch_first=True)
        self.head = nn.Linear(hidden, vocab)

    def forward(self, x, h=None):
        out, h = self.gru(self.emb(x), h)
        return self.head(out), h


torch.manual_seed(0)
lm = CharGRU(len(chars)).to(device)
opt = torch.optim.Adam(lm.parameters(), lr=3e-3)
g = torch.Generator().manual_seed(0)
for _ in range(800):
    starts = torch.randint(0, len(data) - 65, (64,), generator=g)
    x = torch.stack([data[s:s + 64] for s in starts]).to(device)
    y = torch.stack([data[s + 1:s + 65] for s in starts]).to(device)
    opt.zero_grad()
    nn.functional.cross_entropy(lm(x)[0].reshape(-1, len(chars)), y.reshape(-1)).backward()
    opt.step()
lm.eval()


@torch.no_grad()
def beam_search(model, start, length, beam_width):
    ### START CODE HERE ### (≈ 18 dòng)
    # STARTER: # mỗi chùm: (chuỗi chỉ số, tổng log-prob, trạng thái ẩn); khởi tạo bằng start (chạy model qua start)
    # STARTER: # mỗi bước: với mỗi chùm, log_softmax ký tự tiếp theo; tạo ứng viên (score + logp[k]); giữ beam_width tốt nhất
    # STARTER: return None, None
    x = torch.tensor([[stoi[c] for c in start]], device=device)
    logits, h = model(x)
    beams = [([], 0.0, h, logits[:, -1])]
    for _ in range(length):
        candidates = []
        for seq, score, h_b, last_logits in beams:
            logp = torch.log_softmax(last_logits[0], dim=-1)
            top_lp, top_ix = logp.topk(beam_width)
            for lp, ix in zip(top_lp.tolist(), top_ix.tolist()):
                candidates.append((seq + [ix], score + lp, h_b))
        candidates.sort(key=lambda c: c[1], reverse=True)
        beams = []
        for seq, score, h_b in candidates[:beam_width]:
            out, h_new = model(torch.tensor([[seq[-1]]], device=device), h_b)
            beams.append((seq, score, h_new, out[:, -1]))
    best_seq, best_score = beams[0][0], beams[0][1]
    return start + "".join(chars[i] for i in best_seq), best_score
    ### END CODE HERE ###

In [ ]:
results10 = {k: beam_search(lm, "Mua ", 40, k) for k in [1, 3, 8]}
for k, (text, score) in results10.items():
    print(f"beam {k}: log-prob = {score:7.3f} | {text!r}")
check_greater(results10[3][1], results10[1][1] - 1e-6, "beam 3 không tệ hơn greedy về log-prob")
check_greater(results10[8][1], results10[1][1] - 1e-6, "beam 8 không tệ hơn greedy")
check_true(results10[1][0].startswith("Mua ") and len(results10[1][0]) == 44, "độ dài đúng")

**Nhận xét:** Beam search tìm chuỗi có **tổng** log-xác suất cao hơn (hoặc bằng) greedy — greedy chọn ký tự tốt nhất từng bước nhưng có thể dẫn vào nhánh kém về sau. Tuy vậy, chuỗi xác suất cao nhất thường **nhàm chán/lặp lại** (các cụm phổ biến nhất) — vì thế sinh văn bản tự do (chatbot, sáng tác) thường dùng lấy mẫu có nhiệt độ/top-p, còn beam search hợp với bài toán có "đáp án đúng" như dịch máy, nhận dạng tiếng nói. Beam rộng hơn tốn $O(k)$ lần tính toán và có thể ưu tiên chuỗi ngắn (cần chuẩn hoá theo độ dài khi có EOS).